In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Write a GPU program that interleaves two arrays of 32-bit floating point numbers.
  Given two input arrays <code>A</code> and <code>B</code>, each of length <code>N</code>,
  produce an output array of length <code>2N</code> where elements alternate between the two inputs:
  <code>[A[0], B[0], A[1], B[1], A[2], B[2], ...]</code>
</p>

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 400 180" width="400" height="180"
     style="display:block; margin:20px auto;" font-family="monospace" font-size="11">
  <defs>
    <marker id="arrBlue" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 0 L 10 5 L 0 10 z" fill="#4477bb"/>
    </marker>
    <marker id="arrGreen" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 0 L 10 5 L 0 10 z" fill="#44aa66"/>
    </marker>
  </defs>
  <rect width="400" height="180" rx="8" fill="#222"/>

  <!-- Label A -->
  <text x="60" y="20" fill="#ccc" font-size="13" text-anchor="middle" font-weight="bold">A</text>
  <!-- Array A cells -->
  <rect x="10"  y="26" width="40" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="30"  y="45" text-anchor="middle" fill="#ccc">a&#x2080;</text>
  <rect x="54"  y="26" width="40" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="74"  y="45" text-anchor="middle" fill="#ccc">a&#x2081;</text>
  <rect x="98"  y="26" width="40" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="118" y="45" text-anchor="middle" fill="#ccc">a&#x2082;</text>
  <rect x="142" y="26" width="40" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="162" y="45" text-anchor="middle" fill="#ccc">a&#x2083;</text>

  <!-- Label B -->
  <text x="310" y="20" fill="#ccc" font-size="13" text-anchor="middle" font-weight="bold">B</text>
  <!-- Array B cells -->
  <rect x="220" y="26" width="40" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="240" y="45" text-anchor="middle" fill="#ccc">b&#x2080;</text>
  <rect x="264" y="26" width="40" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="284" y="45" text-anchor="middle" fill="#ccc">b&#x2081;</text>
  <rect x="308" y="26" width="40" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="328" y="45" text-anchor="middle" fill="#ccc">b&#x2082;</text>
  <rect x="352" y="26" width="40" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="372" y="45" text-anchor="middle" fill="#ccc">b&#x2083;</text>

  <!-- Label output -->
  <text x="200" y="108" fill="#ccc" font-size="13" text-anchor="middle" font-weight="bold">output</text>
  <!-- Output array cells (alternating blue/green) -->
  <rect x="10"  y="114" width="42" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="31"  y="133" text-anchor="middle" fill="#ccc">a&#x2080;</text>
  <rect x="56"  y="114" width="42" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="77"  y="133" text-anchor="middle" fill="#ccc">b&#x2080;</text>
  <rect x="102" y="114" width="42" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="123" y="133" text-anchor="middle" fill="#ccc">a&#x2081;</text>
  <rect x="148" y="114" width="42" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="169" y="133" text-anchor="middle" fill="#ccc">b&#x2081;</text>
  <rect x="194" y="114" width="42" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="215" y="133" text-anchor="middle" fill="#ccc">a&#x2082;</text>
  <rect x="240" y="114" width="42" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="261" y="133" text-anchor="middle" fill="#ccc">b&#x2082;</text>
  <rect x="286" y="114" width="42" height="28" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="307" y="133" text-anchor="middle" fill="#ccc">a&#x2083;</text>
  <rect x="332" y="114" width="42" height="28" rx="3" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="353" y="133" text-anchor="middle" fill="#ccc">b&#x2083;</text>

  <!-- Curved arrows from A to output (dashed, blue) -->
  <path d="M30,54 C30,80 31,90 31,114" stroke="#4477bb" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrBlue)"/>
  <path d="M74,54 C74,78 123,90 123,114" stroke="#4477bb" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrBlue)"/>
  <path d="M118,54 C118,78 215,90 215,114" stroke="#4477bb" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrBlue)"/>
  <path d="M162,54 C162,78 307,90 307,114" stroke="#4477bb" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrBlue)"/>

  <!-- Curved arrows from B to output (dashed, green) -->
  <path d="M240,54 C240,78 77,90 77,114" stroke="#44aa66" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrGreen)"/>
  <path d="M284,54 C284,78 169,90 169,114" stroke="#44aa66" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrGreen)"/>
  <path d="M328,54 C328,78 261,90 261,114" stroke="#44aa66" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrGreen)"/>
  <path d="M372,54 C372,78 353,90 353,114" stroke="#44aa66" stroke-width="1.2" stroke-dasharray="4,3" fill="none" marker-end="url(#arrGreen)"/>
</svg>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>output</code> array</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:  A = [1.0, 2.0, 3.0], B = [4.0, 5.0, 6.0]
Output: [1.0, 4.0, 2.0, 5.0, 3.0, 6.0]
</pre>

<h2>Example 2:</h2>
<pre>
Input:  A = [10.0, 20.0], B = [30.0, 40.0]
Output: [10.0, 30.0, 20.0, 40.0]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>N</code> &le; 50,000,000</li>

  <li>Performance is measured with <code>N</code> = 25,000,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

__global__ void interleave_kernel(const float* A, const float* B, float* output, int N) {}

// A, B, output are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(const float* A, const float* B, float* output, int N) {
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    interleave_kernel<<<blocksPerGrid, threadsPerBlock>>>(A, B, output, N);
    cudaDeviceSynchronize();
}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# A, B, output are tensors on the GPU
@cute.jit
def solve(A: cute.Tensor, B: cute.Tensor, output: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# A, B are tensors on the GPU
@jax.jit
def solve(A: jax.Array, B: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


def interleave_kernel(
    A: UnsafePointer[Float32, MutExternalOrigin],
    B: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
):
    pass


# A, B, output are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    A: UnsafePointer[Float32, MutExternalOrigin],
    B: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
) raises:
    var BLOCK_SIZE: Int32 = 256
    var ctx = DeviceContext()
    var num_blocks = ceildiv(N, BLOCK_SIZE)

    var _kernel = ctx.compile_function[interleave_kernel, interleave_kernel]()
    ctx.enqueue_function(_kernel, A, B, output, N, grid_dim=num_blocks, block_dim=BLOCK_SIZE)

    ctx.synchronize()


# Torch

In [ ]:
%%writefile solution.py
import torch


# A, B, output are tensors on the GPU
def solve(A: torch.Tensor, B: torch.Tensor, output: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


@triton.jit
def interleave_kernel(A_ptr, B_ptr, output_ptr, N, BLOCK_SIZE: tl.constexpr):
    pass


# A, B, output are tensors on the GPU
def solve(A: torch.Tensor, B: torch.Tensor, output: torch.Tensor, N: int):
    BLOCK_SIZE = 256

    def grid(meta):
        return (triton.cdiv(N, meta["BLOCK_SIZE"]),)

    interleave_kernel[grid](A, B, output, N, BLOCK_SIZE=BLOCK_SIZE)


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Interleave Arrays", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(self, A: torch.Tensor, B: torch.Tensor, output: torch.Tensor, N: int):
        assert A.shape == (N,)
        assert B.shape == (N,)
        assert output.shape == (2 * N,)
        assert A.dtype == B.dtype == output.dtype == torch.float32

        # Interleave: [A[0], B[0], A[1], B[1], ...]
        output[0::2] = A
        output[1::2] = B

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "A": (ctypes.POINTER(ctypes.c_float), "in"),
            "B": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "N": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        A = torch.tensor([1.0, 2.0, 3.0], device="cuda", dtype=dtype)
        B = torch.tensor([4.0, 5.0, 6.0], device="cuda", dtype=dtype)
        output = torch.empty(6, device="cuda", dtype=dtype)
        return {
            "A": A,
            "B": B,
            "output": output,
            "N": 3,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        tests = []

        # Basic example
        tests.append(
            {
                "A": torch.tensor([1.0, 2.0, 3.0], device="cuda", dtype=dtype),
                "B": torch.tensor([4.0, 5.0, 6.0], device="cuda", dtype=dtype),
                "output": torch.empty(6, device="cuda", dtype=dtype),
                "N": 3,
            }
        )

        # Single element
        tests.append(
            {
                "A": torch.tensor([1.0], device="cuda", dtype=dtype),
                "B": torch.tensor([2.0], device="cuda", dtype=dtype),
                "output": torch.empty(2, device="cuda", dtype=dtype),
                "N": 1,
            }
        )

        # Two elements
        tests.append(
            {
                "A": torch.tensor([10.0, 20.0], device="cuda", dtype=dtype),
                "B": torch.tensor([30.0, 40.0], device="cuda", dtype=dtype),
                "output": torch.empty(4, device="cuda", dtype=dtype),
                "N": 2,
            }
        )

        # Negative values
        tests.append(
            {
                "A": torch.tensor([-1.0, -2.0, -3.0], device="cuda", dtype=dtype),
                "B": torch.tensor([-4.0, -5.0, -6.0], device="cuda", dtype=dtype),
                "output": torch.empty(6, device="cuda", dtype=dtype),
                "N": 3,
            }
        )

        # Mixed positive and negative
        tests.append(
            {
                "A": torch.tensor([1.0, -2.0, 3.0, -4.0], device="cuda", dtype=dtype),
                "B": torch.tensor([-1.0, 2.0, -3.0, 4.0], device="cuda", dtype=dtype),
                "output": torch.empty(8, device="cuda", dtype=dtype),
                "N": 4,
            }
        )

        # Zeros
        tests.append(
            {
                "A": torch.zeros(5, device="cuda", dtype=dtype),
                "B": torch.ones(5, device="cuda", dtype=dtype),
                "output": torch.empty(10, device="cuda", dtype=dtype),
                "N": 5,
            }
        )

        # Large values
        tests.append(
            {
                "A": torch.tensor([1e10, 1e-10], device="cuda", dtype=dtype),
                "B": torch.tensor([1e-10, 1e10], device="cuda", dtype=dtype),
                "output": torch.empty(4, device="cuda", dtype=dtype),
                "N": 2,
            }
        )

        # Medium size random
        N = 1024
        tests.append(
            {
                "A": torch.randn(N, device="cuda", dtype=dtype),
                "B": torch.randn(N, device="cuda", dtype=dtype),
                "output": torch.empty(2 * N, device="cuda", dtype=dtype),
                "N": N,
            }
        )

        # Larger random
        N = 10000
        tests.append(
            {
                "A": torch.randn(N, device="cuda", dtype=dtype),
                "B": torch.randn(N, device="cuda", dtype=dtype),
                "output": torch.empty(2 * N, device="cuda", dtype=dtype),
                "N": N,
            }
        )

        # Even larger
        N = 100000
        tests.append(
            {
                "A": torch.randn(N, device="cuda", dtype=dtype),
                "B": torch.randn(N, device="cuda", dtype=dtype),
                "output": torch.empty(2 * N, device="cuda", dtype=dtype),
                "N": N,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        N = 25000000  # 25 million elements each, 50 million output
        return {
            "A": torch.randn(N, device="cuda", dtype=dtype),
            "B": torch.randn(N, device="cuda", dtype=dtype),
            "output": torch.empty(2 * N, device="cuda", dtype=dtype),
            "N": N,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
